# SORA 2 Video Generation - Complete Demo

This notebook demonstrates all three video generation capabilities of SORA 2 in Azure OpenAI:
1. **Text-to-Video**: Generate videos from text prompts
2. **Image-to-Video**: Create videos from reference images
3. **Video-to-Video**: Transform existing videos with new prompts (including remix capability)

## Prerequisites

- Azure OpenAI resource with SORA 2 model deployed
- Latest OpenAI Python SDK: `pip install openai --upgrade`
- Create a `.env` file in this directory with your credentials:

  ```  ```

  AZURE_OPENAI_ENDPOINT=https://your-resource.openai.azure.com/  AZURE_OPENAI_DEPLOYMENT_NAME=sora-2
  AZURE_OPENAI_API_KEY=your-api-key

## Setup and Configuration

First, let's install required packages and set up authentication.

In [ ]:
# Install/upgrade required packages
#%pip install openai --upgrade
#%pip install python-dotenv
#%pip install azure-identity

In [1]:
import os
import time
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Video, display, HTML
import requests
from io import BytesIO

# Load environment variables from .env file in the current directory
load_dotenv('.env')

# Configuration - Load from environment variables
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
SORA_MODEL_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME", "sora-2")

# Validate required environment variables
if not AZURE_OPENAI_ENDPOINT:
    raise ValueError("AZURE_OPENAI_ENDPOINT not found in .env file")
if not AZURE_OPENAI_API_KEY:
    raise ValueError("AZURE_OPENAI_API_KEY not found in .env file")

# Create output directories
Path("videos").mkdir(exist_ok=True)
Path("images").mkdir(exist_ok=True)

print("✅ Configuration loaded from .env file")
print(f"Endpoint: {AZURE_OPENAI_ENDPOINT}")
print(f"Model Deployment: {SORA_MODEL_DEPLOYMENT}")

✅ Configuration loaded from .env file
Endpoint: https://aq-ai-foundry-sweden-central.openai.azure.com/openai/v1/
Model Deployment: sora-2


In [2]:
# Initialize OpenAI client
client = OpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    base_url=AZURE_OPENAI_ENDPOINT,
)

print("✅ OpenAI client initialized")

✅ OpenAI client initialized


## Helper Functions

Let's create helper functions for video generation, polling, and downloading.

In [3]:
def create_video_job(prompt, model=None, size="720x1280", seconds=4, input_reference=None, remix_video_id=None):
    """
    Create a video generation job
    
    Args:
        prompt: Natural language description of the video
        model: Model name (defaults to SORA_MODEL_DEPLOYMENT)
        size: Video resolution (720x1280, 1280x720, 1024x1792, 1792x1024)
        seconds: Duration (4, 8, or 12 seconds)
        input_reference: Optional image/video file for reference
        remix_video_id: Optional video ID to remix
    
    Returns:
        Video object with job details
    """
    if model is None:
        model = SORA_MODEL_DEPLOYMENT
    
    kwargs = {
        "model": model,
        "prompt": prompt,
        "size": size,
        "seconds": str(seconds),  # Must be a string: "4", "8", or "12"
    }
    
    if input_reference is not None:
        kwargs["input_reference"] = input_reference
    
    if remix_video_id is not None:
        kwargs["remix_video_id"] = remix_video_id
    
    video = client.videos.create(**kwargs)
    
    print(f"🎬 Video generation started!")
    print(f"   ID: {video.id}")
    print(f"   Status: {video.status}")
    print(f"   Model: {video.model}")
    print(f"   Size: {video.size}")
    print(f"   Duration: {video.seconds}s")
    
    return video


def poll_video_status(video_id, poll_interval=20, max_wait_time=600):
    """
    Poll video generation status until completion
    
    Args:
        video_id: Video job ID
        poll_interval: Seconds between status checks
        max_wait_time: Maximum time to wait in seconds
    
    Returns:
        Final video object
    """
    print(f"\n⏳ Polling video status (checking every {poll_interval}s)...")
    
    start_time = time.time()
    video = client.videos.retrieve(video_id)
    
    while video.status not in ["completed", "failed", "cancelled"]:
        elapsed = int(time.time() - start_time)
        
        if elapsed > max_wait_time:
            print(f"⚠️ Max wait time ({max_wait_time}s) exceeded")
            break
        
        print(f"   [{elapsed}s] Status: {video.status} | Progress: {video.progress}%")
        time.sleep(poll_interval)
        video = client.videos.retrieve(video_id)
    
    # Final status
    if video.status == "completed":
        print(f"\n✅ Video successfully completed!")
        print(f"   Completed at: {video.completed_at}")
        print(f"   Expires at: {video.expires_at}")
    else:
        print(f"\n❌ Video generation ended with status: {video.status}")
        if video.error:
            print(f"   Error: {video.error}")
    
    return video


def download_video(video_id, output_path):
    """
    Download completed video
    
    Args:
        video_id: Video job ID
        output_path: Path to save the video file
    
    Returns:
        Path to downloaded file
    """
    print(f"\n📥 Downloading video...")
    
    content = client.videos.download_content(video_id, variant="video")
    content.write_to_file(output_path)
    
    file_size = os.path.getsize(output_path) / (1024 * 1024)  # MB
    print(f"✅ Video saved to: {output_path}")
    print(f"   File size: {file_size:.2f} MB")
    
    return output_path


def display_video_inline(video_path, width=640):
    """Display video inline in the notebook"""
    display(Video(video_path, width=width, embed=True))


def generate_and_download_video(prompt, output_filename, **kwargs):
    """
    Complete workflow: create, poll, and download video
    
    Args:
        prompt: Video description
        output_filename: Name for output file (will be saved in videos/)
        **kwargs: Additional arguments for create_video_job
    
    Returns:
        Path to downloaded video
    """
    print("=" * 70)
    print(f"🎥 GENERATING VIDEO: {output_filename}")
    print("=" * 70)
    print(f"Prompt: {prompt}")
    print()
    
    # Create job
    video = create_video_job(prompt, **kwargs)
    
    # Poll until complete
    video = poll_video_status(video.id)
    
    # Download if successful
    if video.status == "completed":
        output_path = f"videos/{output_filename}"
        download_video(video.id, output_path)
        return output_path, video.id
    else:
        print(f"\n❌ Video generation failed")
        return None, video.id

print("✅ Helper functions defined")

✅ Helper functions defined


---

## Demo 1: Text-to-Video Generation

Generate videos directly from text prompts. This is the most straightforward use case.

### Best Practices for Prompts:
- Use natural language descriptions
- Include shot type, subject, action, setting, and lighting
- Specify camera motion if desired
- Keep prompts single-purpose for best adherence
- Write in English or Latin script languages

In [ ]:
# Example 1: Simple text-to-video
prompt_1 = "A cool cat wearing sunglasses riding a motorcycle through a neon-lit city street at night"

video_path_1, video_id_1 = generate_and_download_video(
    prompt=prompt_1,
    output_filename="demo1_text_to_video_cat.mp4",
    size="1280x720",  # Landscape
    seconds=8
)

In [ ]:
# Display the generated video
if video_path_1:
    print("\n🎬 Displaying generated video:")
    display_video_inline(video_path_1)

In [ ]:
# Example 2: More complex scene with camera movement
prompt_2 = "A cinematic drone shot flying over a tropical beach at sunset, waves crashing on the shore, palm trees swaying, golden hour lighting"

video_path_2, video_id_2 = generate_and_download_video(
    prompt=prompt_2,
    output_filename="demo1_text_to_video_beach.mp4",
    size="1280x720",  # Landscape
    seconds=8  # Longer duration
)

In [ ]:
# Display the generated video
if video_path_2:
    print("\n🎬 Displaying generated video:")
    display_video_inline(video_path_2)

---

## Demo 2: Image-to-Video Generation

Generate videos from reference images. The image serves as the first frame or visual anchor.

### Important Notes:
- The reference image resolution must match the output video resolution
- Supported formats: JPEG, PNG, WebP
- Use descriptive prompts to guide the animation

In [4]:
# First, let's check what images we have available
print("Available images in the images/ directory:")
images_dir = Path("images")
if images_dir.exists():
    image_files = list(images_dir.glob("*"))
    for img in image_files:
        if img.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']:
            print(f"  - {img.name}")
else:
    print("  No images directory found. Create one and add sample images.")

Available images in the images/ directory:
  - blue-jay.png


### Option A: AI-Powered Image Description + Video Generation

Use GPT-4o to analyze an image and automatically generate a descriptive prompt for video generation:

In [5]:
# AI-powered image description and video generation
# This approach uses GPT-4 Vision to describe the image and create a video prompt

import base64
from PIL import Image

def get_image_dimensions(image_path):
    """Get the dimensions of an image file"""
    with Image.open(image_path) as img:
        return img.size  # Returns (width, height)

def get_video_size_from_image(image_path):
    """
    Determine the appropriate video size parameter based on image dimensions
    
    Returns:
        Tuple of (size_string, width, height) or None if dimensions don't match supported sizes
    """
    width, height = get_image_dimensions(image_path)
    
    # Map of supported video sizes (width, height)
    supported_sizes = {
        "720x1280": (720, 1280),   # Portrait
        "1280x720": (1280, 720),   # Landscape
        "1024x1792": (1024, 1792), # Tall portrait
        "1792x1024": (1792, 1024), # Wide landscape
    }
    
    # Check if image matches any supported size
    for size_str, (w, h) in supported_sizes.items():
        if width == w and height == h:
            return size_str, width, height
    
    # If no exact match, return None
    return None, width, height

def resize_image_for_video(input_path, output_path=None, target_size="720x1280"):
    """
    Resize an image to match supported video dimensions
    
    Args:
        input_path: Path to input image
        output_path: Path to save resized image (if None, creates a new file)
        target_size: One of "720x1280", "1280x720", "1024x1792", "1792x1024"
    
    Returns:
        Path to resized image
    """
    # Parse target dimensions
    target_width, target_height = map(int, target_size.split('x'))
    
    # Open and resize image
    with Image.open(input_path) as img:
        # Determine if we should crop or pad to maintain aspect ratio
        orig_width, orig_height = img.size
        orig_aspect = orig_width / orig_height
        target_aspect = target_width / target_height
        
        if orig_aspect > target_aspect:
            # Image is wider - crop width
            new_width = int(orig_height * target_aspect)
            left = (orig_width - new_width) // 2
            img_cropped = img.crop((left, 0, left + new_width, orig_height))
        else:
            # Image is taller - crop height
            new_height = int(orig_width / target_aspect)
            top = (orig_height - new_height) // 2
            img_cropped = img.crop((0, top, orig_width, top + new_height))
        
        # Resize to exact target dimensions
        img_resized = img_cropped.resize((target_width, target_height), Image.Resampling.LANCZOS)
        
        # Generate output path if not provided
        if output_path is None:
            input_stem = Path(input_path).stem
            input_dir = Path(input_path).parent
            output_path = input_dir / f"{input_stem}_resized_{target_size}.png"
        
        # Save resized image
        img_resized.save(output_path)
        print(f"✅ Image resized from {orig_width}x{orig_height} to {target_width}x{target_height}")
        print(f"   Saved to: {output_path}")
        
        return str(output_path)

def encode_image(image_path):
    """Encode image to base64 for GPT-4 Vision API"""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

def describe_image_for_video(image_path, style_hint=""):
    """
    Use GPT-4 Vision to analyze an image and generate a video prompt
    
    Args:
        image_path: Path to the image file
        style_hint: Optional style guidance (e.g., "cinematic", "animated", "realistic")
    
    Returns:
        Generated prompt for video creation
    """
    # Encode the image
    base64_image = encode_image(image_path)
    
    # Create a vision-capable client
    vision_messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": f"""Analyze this image and create a detailed prompt for video generation. 
                    
The prompt should:
1. Describe what you see in the image
2. Suggest natural, realistic movements or animations
3. Include lighting, mood, and atmosphere details
4. Keep it concise but descriptive (2-3 sentences max)
{f'5. Apply this style: {style_hint}' if style_hint else ''}

Format: Just return the video prompt, nothing else."""
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{base64_image}"
                    }
                }
            ]
        }
    ]
    
    # Call GPT-4 Vision
    response = client.chat.completions.create(
        model="gpt-4o",  # Update with your GPT-4 Vision deployment name if different
        messages=vision_messages,
        max_tokens=300
    )
    
    return response.choices[0].message.content.strip()

# Example: Automatically describe and animate an image
ai_image_filename = "blue-jay.png"  # Change this to your image
ai_image_path = f"images/{ai_image_filename}"

if Path(ai_image_path).exists():
    print(f"🔍 Analyzing image: {ai_image_filename}")
    print("=" * 70)
    
    # Check image dimensions
    video_size, img_width, img_height = get_video_size_from_image(ai_image_path)
    print(f"📐 Image dimensions: {img_width}x{img_height}")
    
    if video_size is None:
        print(f"⚠️ Image dimensions {img_width}x{img_height} don't match supported video sizes!")
        print(f"   Supported sizes: 720x1280, 1280x720, 1024x1792, 1792x1024")
        print()
        
        # Automatically resize to the closest portrait or landscape format
        img_aspect = img_width / img_height
        if img_aspect < 1:  # Portrait
            target_size = "720x1280"
        else:  # Landscape
            target_size = "1280x720"
        
        print(f"🔧 Automatically resizing to {target_size}...")
        ai_image_path = resize_image_for_video(ai_image_path, target_size=target_size)
        video_size = target_size
        print()
    else:
        print(f"✅ Image size matches video format: {video_size}")
        print()
    
    # Get AI-generated description and video prompt
    generated_prompt = describe_image_for_video(
        ai_image_path,
        style_hint="cinematic, professional quality"  # Optional style guidance
    )
    
    print(f"✨ AI-Generated Prompt:")
    print(f"   {generated_prompt}")
    print()
    
    # Generate video with AI-created prompt and matching dimensions
    ai_video_path, ai_video_id = generate_and_download_video(
        prompt=generated_prompt,
        output_filename="demo2_ai_described_video.mp4",
        size=video_size,  # Use the matched video size
        seconds=8,
        input_reference=open(ai_image_path, "rb")
    )
else:
    print(f"⚠️ Image not found: {ai_image_path}")
    print("Please add an image to the images/ directory and update the filename above.")
    ai_video_path = None

🔍 Analyzing image: blue-jay.png
📐 Image dimensions: 707x990
⚠️ Image dimensions 707x990 don't match supported video sizes!
   Supported sizes: 720x1280, 1280x720, 1024x1792, 1792x1024

🔧 Automatically resizing to 720x1280...
✅ Image resized from 707x990 to 720x1280
   Saved to: images/blue-jay_resized_720x1280.png

✨ AI-Generated Prompt:
   A majestic blue jay wearing a Toronto Blue Jays jersey perches proudly atop the Seattle Space Needle, its sharp gaze scanning the city skyline. Its wings slowly unfold, creating a dynamic gust that ripples through the fabric of its jersey, amidst a backdrop of dramatic stormy clouds and diffused cinematic lighting. The atmosphere is tense yet inspiring, capturing the essence of a legendary guardian overlooking an urban expanse.

🎥 GENERATING VIDEO: demo2_ai_described_video.mp4
Prompt: A majestic blue jay wearing a Toronto Blue Jays jersey perches proudly atop the Seattle Space Needle, its sharp gaze scanning the city skyline. Its wings slowly unfold

In [6]:
# Display the AI-generated video
if ai_video_path:
    print("\n🎬 Displaying AI-described video:")
    display_video_inline(ai_video_path)


🎬 Displaying AI-described video:


### Option B: Manual Image-to-Video from Local File

If you prefer to write your own prompt:

In [ ]:
# Example with local image file
# Replace 'your_image.jpg' with an actual image from your images/ directory

image_filename = "blue-jay.png"  # Change this to your image
image_path = f"images/{image_filename}"

# Check if image exists
if Path(image_path).exists():
    # Check image dimensions first
    video_size, img_width, img_height = get_video_size_from_image(image_path)
    print(f"📐 Image dimensions: {img_width}x{img_height}")
    
    if video_size is None:
        print(f"⚠️ Image dimensions {img_width}x{img_height} don't match supported video sizes!")
        print(f"   Supported sizes: 720x1280, 1280x720, 1024x1792, 1792x1024")
        print()
        
        # Automatically resize to the closest portrait or landscape format
        img_aspect = img_width / img_height
        if img_aspect < 1:  # Portrait
            target_size = "720x1280"
        else:  # Landscape
            target_size = "1280x720"
        
        print(f"🔧 Automatically resizing to {target_size}...")
        image_path = resize_image_for_video(image_path, target_size=target_size)
        video_size = target_size
        print()
    else:
        print(f"✅ Image size matches video format: {video_size}")
    
    prompt_3 = "A mean looking blue Jay wearing the toronto blue jays shirt, flapping its wings on top of the seattle needle, natural lighting, professional video quality"
    
    with open(image_path, "rb") as image_file:
        video_path_3, video_id_3 = generate_and_download_video(
            prompt=prompt_3,
            output_filename="demo2_image_to_video_local.mp4",
            size=video_size,  # Use the matched video size
            seconds=8,
            input_reference=image_file
        )
else:
    print(f"⚠️ Image not found: {image_path}")
    print("Please add an image to the images/ directory and update the filename above.")
    video_path_3 = None

In [ ]:
# Display the generated video
if video_path_3:
    print("\n🎬 Displaying generated video from image:")
    display_video_inline(video_path_3)

### Option B: Image-to-Video from URL

You can also use images from URLs:

In [ ]:
# Example with image URL
# This is a sample URL - replace with your own image URL

image_url = "https://example.com/your-image.jpg"  # Replace with actual URL

# Uncomment and modify the code below when you have a valid image URL:

"""
prompt_4 = "The scene comes to life with gentle movement, birds flying in the background, soft wind effects"

# Download image from URL
response = requests.get(image_url)
if response.status_code == 200:
    image_data = BytesIO(response.content)
    image_data.name = "reference_image.jpg"
    
    video_path_4, video_id_4 = generate_and_download_video(
        prompt=prompt_4,
        output_filename="demo2_image_to_video_url.mp4",
        size="1280x720",
        seconds=8,
        input_reference=image_data
    )
    
    if video_path_4:
        print("\\n🎬 Displaying generated video from URL image:")
        display_video_inline(video_path_4)
else:
    print(f"❌ Failed to download image from URL: {response.status_code}")
"""

print("💡 Uncomment the code above and provide a valid image URL to test this feature")

---

## Demo 3: Video-to-Video Generation

Transform existing videos using reference video inputs. This feature is powerful for video editing and transformation.

### Important Notes:
- The reference video resolution must match the output video resolution
- Input videos can be used to guide structure, motion, and framing
- Use the `remix_video_id` parameter to remix previously generated videos

In [ ]:
# First, let's check what videos we have available
print("Available videos in the inputvideos/ directory:")
input_videos_dir = Path("inputvideos")
if input_videos_dir.exists():
    video_files = list(input_videos_dir.glob("*.mp4"))
    for vid in video_files:
        print(f"  - {vid.name}")
else:
    print("  No inputvideos directory found. Create one and add sample videos.")

### Option A: Video-to-Video from Local File

Transform a local video with a new prompt:

In [ ]:
# Example with local video file
# Replace 'your_video.mp4' with an actual video from your inputvideos/ directory

input_video_filename = "sample_video.mp4"  # Change this to your video
input_video_path = f"inputvideos/{input_video_filename}"

# Check if video exists
if Path(input_video_path).exists():
    prompt_5 = "Transform the scene into a vibrant animated style with bold colors and smooth motion"
    
    with open(input_video_path, "rb") as video_file:
        video_path_5, video_id_5 = generate_and_download_video(
            prompt=prompt_5,
            output_filename="demo3_video_to_video_local.mp4",
            size="1280x720",  # Must match input video dimensions
            seconds=8,
            input_reference=video_file
        )
else:
    print(f"⚠️ Video not found: {input_video_path}")
    print("Please add a video to the inputvideos/ directory and update the filename above.")
    video_path_5 = None

In [ ]:
# Display the generated video
if video_path_5:
    print("\n🎬 Displaying transformed video:")
    display_video_inline(video_path_5)

### Option B: Remix Previously Generated Video

The SORA 2 API includes a powerful remix feature that allows you to make targeted adjustments to previously generated videos without regenerating from scratch.

**Note:** There is currently a known issue with remix functionality in Azure OpenAI. The commands execute successfully, but the generated video may not be based on the previously generated video as expected. This will be resolved in a future update.

In [ ]:
# Remix a previously generated video
# Use the video ID from any of the videos we generated earlier

# Example: Remix the first video we created (the cat on motorcycle)
if 'video_id_1' in locals() and video_id_1:
    prompt_6 = "Same scene but now it's raining heavily with dramatic lighting and reflections on wet streets"
    
    video_path_6, video_id_6 = generate_and_download_video(
        prompt=prompt_6,
        output_filename="demo3_video_remix.mp4",
        size="1280x720",
        seconds=4,
        remix_video_id=video_id_1  # Reference to previous video
    )
    
    if video_path_6:
        print("\n🎬 Displaying remixed video:")
        display_video_inline(video_path_6)
else:
    print("⚠️ No previous video ID available for remix. Generate a video in Demo 1 first.")

---

## Advanced Features

### List All Generated Videos

You can list all videos generated in your session:

In [ ]:
# List videos (with pagination)
try:
    videos_list = client.videos.list(limit=10)
    
    print("📋 Recent videos:")
    print("=" * 70)
    
    for video in videos_list.data:
        status_emoji = "✅" if video.status == "completed" else "⏳" if video.status in ["queued", "in_progress"] else "❌"
        print(f"{status_emoji} ID: {video.id}")
        print(f"   Status: {video.status} | Progress: {video.progress}%")
        print(f"   Size: {video.size} | Duration: {video.seconds}s")
        print(f"   Created: {video.created_at}")
        if video.completed_at:
            print(f"   Completed: {video.completed_at}")
        print()
        
except Exception as e:
    print(f"⚠️ Could not list videos: {e}")

### Delete a Video

Videos are stored for 24 hours. You can delete them earlier if needed:

In [ ]:
# Example: Delete a specific video by ID
# Uncomment and replace with actual video ID to delete

"""
video_id_to_delete = "video_12345..."

try:
    client.videos.delete(video_id_to_delete)
    print(f"✅ Video {video_id_to_delete} deleted successfully")
except Exception as e:
    print(f"❌ Error deleting video: {e}")
"""

print("💡 Uncomment the code above to delete a specific video")

---

## Summary and Key Takeaways

### SORA 2 Capabilities Demonstrated:

1. **Text-to-Video**: Create videos from descriptive prompts
   - Simple scenes to complex cinematography
   - Various resolutions and durations

2. **Image-to-Video**: Animate static images
   - Local files or URLs
   - Match image and video resolutions

3. **Video-to-Video**: Transform and remix videos
   - Style transfer and modifications
   - Remix feature for iterative adjustments

### Technical Specifications:

- **Supported Resolutions**: 720×1280 (portrait), 1280×720 (landscape), 1024×1792, 1792×1024
- **Durations**: 4, 8, or 12 seconds
- **Concurrent Jobs**: Maximum 2 jobs at once
- **Availability**: 24 hours after creation
- **Audio**: Sora 2 supports audio generation in output videos

### Best Practices:

1. **Prompting**: Be specific about shot type, subject, action, setting, lighting, and camera motion
2. **Resolution Matching**: When using reference images/videos, ensure dimensions match output
3. **Polling**: Use appropriate intervals (20s recommended) to avoid excessive API calls
4. **Error Handling**: Always check job status before attempting to download
5. **Content Safety**: Sora includes content filtering and safety classifiers

### Limitations:

- Complex physics may not be accurately represented
- Causal relationships (e.g., bite marks on a cookie) can be challenging
- Spatial reasoning (left vs. right) may have issues
- Precise time-based event sequencing may vary

---

**🎉 You've completed the SORA 2 demo notebook!**

Experiment with different prompts, images, and videos to explore the full capabilities of SORA 2 video generation.